# Notebook 01: LangGraph Agentic RAG Extraction + Verification Pipeline

**Purpose:** Run the full multi-agent LangGraph extraction pipeline over deidentified OCR text.  
Extracts all 13 clinical features, verifies each via RAG, flags fabrications, and produces auditable JSON output.

## Pipeline Architecture
```
load_case → index_case → next_feature → retrieve → extract → verify
  verify → (adjudicate | self_consistency | rewrite_query)
  rewrite_query → retrieve_again → extract → verify
  self_consistency → adjudicate
  adjudicate → (next_feature | aggregate) → END
```

## Features Extracted (13 total)
1. Lesion Size  
2. Lesion Location  
3. Calcifications / Asymmetry  
4. Additional Enhancement (MRI)  
5. Disease Extent  
6. Clip Placement  
7. Workup Recommendation  
8. Lymph Node Findings  
9. Chronology Preserved  
10. Biopsy Method  
11. Invasive Component Size (Pathology) ← high-risk  
12. Histologic Diagnosis  
13. Receptor Status (ER/PR/HER2) ← high-risk  

**References:**  
- `src/workflows/extraction_graph.py` — LangGraph graph  
- `src/agents/` — 5 specialized agents  
- `src/rag/feature_queries.py` — feature registry  
- `models/configs/safety_thresholds.yaml` — verification thresholds

## 0. Environment Setup

In [ ]:
import os
import sys
import json
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

PROJECT_ROOT = Path(
    os.getenv(
        "PROJECT_ROOT",
        r"C:\Users\jamesr4\OneDrive - Memorial Sloan Kettering Cancer Center"
        r"\Documents\GitHub\llm_summarization_br_ca",
    )
)
DATA_PRIVATE_DIR = Path(
    os.getenv("DATA_PRIVATE_DIR", r"C:\Users\jamesr4\loc\data_private")
)

sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_PRIVATE_DIR: {DATA_PRIVATE_DIR}")
print(f"ANTHROPIC_API_KEY set: {'ANTHROPIC_API_KEY' in os.environ}")

## 1. Install / verify dependencies

In [ ]:
# !uv add langgraph langchain-anthropic langchain-community langchain-huggingface
# !uv add langchain-text-splitters faiss-cpu sentence-transformers
# !uv add networkx pydantic pyyaml python-dotenv pandas pyarrow

import langgraph
import langchain_anthropic
import langchain_community
print(f"langgraph version: {langgraph.__version__}")
print(f"langchain_anthropic version: {langchain_anthropic.__version__}")

## 2. Feature Registry — inspect all 13 features

In [ ]:
import pandas as pd
from src.rag.feature_queries import FEATURES, CRITICAL_FEATURES, HIGH_RISK_SELF_CONSISTENCY

feat_df = pd.DataFrame([
    {
        "feature": k,
        "display": v["display_name"],
        "k": v["k"],
        "k_second_pass": v["k_second_pass"],
        "critical": v["critical"],
        "verification_threshold": v["verification_threshold"],
    }
    for k, v in FEATURES.items()
])

print(f"Total features: {len(FEATURES)}")
print(f"Critical features: {CRITICAL_FEATURES}")
print(f"Self-consistency features: {HIGH_RISK_SELF_CONSISTENCY}")
feat_df

## 3. Load OCR Text — from DATA_PRIVATE_DIR

In [ ]:
extracted_text_dir = DATA_PRIVATE_DIR / "extracted_text"

txt_files = sorted(extracted_text_dir.glob("*.txt")) if extracted_text_dir.exists() else []
print(f"Found {len(txt_files)} extracted text files")

def load_ocr_text(path: Path) -> str:
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        return f.read()

# Preview first case
if txt_files:
    sample_path = txt_files[0]
    sample_text = load_ocr_text(sample_path)
    print(f"Sample case: {sample_path.stem}")
    print(f"Characters: {len(sample_text)}")
    print("First 500 chars:")
    print(sample_text[:500])
else:
    print("No text files found in DATA_PRIVATE_DIR/extracted_text")
    print("Using DEMO text for illustration purposes.")
    sample_text = (
        "MAMMOGRAPHY (2023-01-10): Diagnostic bilateral. 1.5 cm irregular mass, "
        "left breast, 10 o'clock, 4 cm from nipple. BI-RADS 4C. "
        "ULTRASOUND (2023-01-15): Left breast mass 1.5 x 1.2 x 1.0 cm at 10 o'clock. "
        "US-guided core biopsy performed. Tumark clip placed. "
        "PATHOLOGY: Invasive ductal carcinoma, grade 2. Invasive component 1.3 cm. "
        "ER: Positive (90%, strong). PR: Positive (60%). HER2 IHC: 1+. "
        "HER2 ISH: Not performed. "
        "LYMPH NODES: No axillary lymphadenopathy identified."
    )
    sample_path = type('obj', (object,), {'stem': 'DEMO_CASE_001'})()

## 4. Chunk and Index OCR Text

In [ ]:
from src.preprocessing.chunk_text import chunk_ocr_text

case_id = sample_path.stem
chunks = chunk_ocr_text(sample_text, case_id=case_id)

print(f"Case: {case_id}")
print(f"Total chunks: {len(chunks)}")
for i, c in enumerate(chunks[:3], 1):
    print(f"\n--- Chunk {i} | modality={c['modality']} | tokens={c['token_count']} ---")
    print(c["text"][:200])

## 5. Build Vector Index

In [ ]:
from src.rag.embed_chunks import build_faiss_index, get_embedder

print("Building FAISS index (this may take 30-60s on first run)...")
embedder = get_embedder()
index = build_faiss_index(chunks, embedder)
print(f"Index built: {index.index.ntotal} vectors")

## 6. Test Feature Retrieval

In [ ]:
from src.rag.retrievers import retrieve_for_feature, get_feature_query, format_chunks_for_prompt

test_feature = "feature_13_receptor_status"
query = get_feature_query(test_feature)
retrieved = retrieve_for_feature(index, case_id, test_feature, query, k=5)

print(f"Feature: {test_feature}")
print(f"Query: {query}")
print(f"Retrieved: {len(retrieved)} chunks")
print("\n" + format_chunks_for_prompt(retrieved))

## 7. Run Full LangGraph Pipeline — Single Case

In [ ]:
from src.workflows.orchestration import run_single_case
from src.utils.io_utils import generate_run_id

run_id = generate_run_id()
print(f"Run ID: {run_id}")
print(f"Case: {case_id}")
print("Starting pipeline...")

result = run_single_case(
    case_id=case_id,
    ocr_text=sample_text,
    prompt_id="rag_verify_v1",
    model_id="claude-3-5-sonnet-20241022",
    run_id=run_id,
)

print("\nPipeline complete!")
print(f"Features extracted: {len(result['features'])}")
print(f"Fabrication flags: {result['fabrication_flags']}")
print(f"Omission flags: {result['omission_flags']}")

## 8. Inspect Feature-Level Results

In [ ]:
import pandas as pd

rows = []
for feat_name, feat_data in result["features"].items():
    rows.append({
        "feature": feat_name,
        "value": feat_data.get("value", ""),
        "confidence": feat_data.get("confidence", 0.0),
        "supported": feat_data.get("supported"),
        "verification_confidence": feat_data.get("verification_confidence"),
        "verdict": feat_data.get("verdict"),
        "retrieval_attempts": feat_data.get("retrieval_attempts", 0),
        "verification_method": feat_data.get("verification_method"),
    })

results_df = pd.DataFrame(rows)
results_df

## 9. Verdict Distribution

In [ ]:
import matplotlib.pyplot as plt

verdict_counts = results_df["verdict"].value_counts()

colors = {
    "CORRECT": "#2ecc71",
    "FABRICATION": "#e74c3c",
    "OMISSION": "#f39c12",
    "UNCERTAIN": "#95a5a6",
}

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    verdict_counts.index,
    verdict_counts.values,
    color=[colors.get(v, "#bdc3c7") for v in verdict_counts.index],
    edgecolor="white",
    linewidth=1.5,
)
ax.set_title(f"Feature Verdict Distribution — Case {case_id}", fontsize=13, fontweight="bold")
ax.set_ylabel("Count")
ax.set_xlabel("Verdict")
for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.05,
        str(int(bar.get_height())),
        ha="center", va="bottom", fontweight="bold",
    )
ax.set_ylim(0, verdict_counts.max() + 2)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / f"{case_id}_verdict_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Fabrication rate: {(verdict_counts.get('FABRICATION', 0) / len(results_df)):.1%}")

## 10. Confidence Distribution by Feature

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

colors_conf = [colors.get(v, "#bdc3c7") for v in results_df["verdict"]]

bars = ax.barh(
    results_df["feature"].str.replace("feature_", "").str.replace("_", " "),
    results_df["confidence"].fillna(0),
    color=colors_conf,
    edgecolor="white",
    linewidth=1.2,
)
ax.axvline(x=0.75, color="black", linestyle="--", alpha=0.5, label="Threshold 0.75")
ax.set_xlabel("Extraction Confidence")
ax.set_title("Extraction Confidence by Feature", fontsize=13, fontweight="bold")
ax.set_xlim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / f"{case_id}_confidence_by_feature.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Verification Pass Rate

In [ ]:
supported = results_df["supported"].dropna()
pass_rate = supported.mean() if len(supported) > 0 else 0.0

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Verification support pie
support_counts = results_df["supported"].value_counts(dropna=False)
labels = ["Supported" if v is True else "Not Verifiable" if v is False else "Skipped"
          for v in support_counts.index]
axes[0].pie(
    support_counts.values,
    labels=labels,
    colors=["#2ecc71", "#e74c3c", "#95a5a6"],
    autopct="%1.1f%%",
    startangle=90,
)
axes[0].set_title("Verification Support Status", fontweight="bold")

# Retrieval attempts
attempt_counts = results_df["retrieval_attempts"].value_counts().sort_index()
axes[1].bar(
    attempt_counts.index.astype(str),
    attempt_counts.values,
    color="#3498db",
    edgecolor="white",
)
axes[1].set_title("Retrieval Attempts per Feature", fontweight="bold")
axes[1].set_xlabel("Attempts")
axes[1].set_ylabel("Count")

plt.suptitle(f"Verification & Retrieval — Case {case_id}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / f"{case_id}_verification_stats.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Verification pass rate: {pass_rate:.1%}")

## 12. Save Structured Case Output

In [ ]:
from src.utils.io_utils import save_json

out_dir = PROJECT_ROOT / "data" / "processed" / "feature_outputs"
out_path = out_dir / f"{run_id}_{case_id}.json"
save_json(result, out_path)
print(f"Saved: {out_path}")

# Pretty-print key feature outputs
print("\n=== RECEPTOR STATUS ===")
r13 = result["features"].get("feature_13_receptor_status", {})
print(f"  Value: {r13.get('value')}")
print(f"  Verdict: {r13.get('verdict')}")
print(f"  Confidence: {r13.get('confidence')}")
print(f"  Supported: {r13.get('supported')}")
print(f"  Quote: {r13.get('verification_quote')}")

print("\n=== INVASIVE COMPONENT SIZE ===")
r11 = result["features"].get("feature_11_invasive_component_size_pathology", {})
print(f"  Value: {r11.get('value')}")
print(f"  Verdict: {r11.get('verdict')}")
print(f"  Method: {r11.get('verification_method')}")

## 13. Build Knowledge Graph from Results

In [ ]:
from src.graph.build_graph import results_to_kg, build_networkx_graph, save_graphml
from src.graph.kg_retriever import summarize_graph_stats, get_fabrications

kg = results_to_kg([result])
G = build_networkx_graph(kg)

stats = summarize_graph_stats(G)
print("Knowledge Graph Stats:")
for k, v in stats.items():
    print(f"  {k}: {v}")

fabrications = get_fabrications(G)
print(f"\nFabrication nodes in graph: {len(fabrications)}")
for f in fabrications:
    print(f"  {f.get('feature_name')}: {f.get('value')}")

graphml_path = PROJECT_ROOT / "data" / "knowledge_graph" / f"{run_id}_kg.graphml"
save_graphml(G, graphml_path)
print(f"\nGraphML saved: {graphml_path}")

## 14. Batch Run (Multi-Case)

In [ ]:
# Load multiple cases from DATA_PRIVATE_DIR
max_cases = 5  # adjust as needed

cases = []
for txt_path in txt_files[:max_cases]:
    cases.append({
        "case_id": txt_path.stem,
        "ocr_text": load_ocr_text(txt_path),
    })

if not cases:
    print("No cases found in DATA_PRIVATE_DIR — using demo case only.")
else:
    from src.workflows.orchestration import run_batch
    batch_run_id = generate_run_id()
    print(f"Starting batch run: {batch_run_id} ({len(cases)} cases)")
    # batch_results = run_batch(
    #     cases=cases,
    #     prompt_id="rag_verify_v1",
    #     model_id="claude-3-5-sonnet-20241022",
    #     run_id=batch_run_id,
    #     save_results=True,
    # )
    print("Uncomment run_batch() to execute. Outputs saved to experiments/runs/")

## 15. Compute HCAT Safety Metrics

In [ ]:
from eval.metrics.hcat_metrics import compute_hcat_score, hcat_report_to_df
from eval.schemas.hcat_schema import HCATBatchReport

results_df["case_id"] = case_id
score = compute_hcat_score(results_df, case_id=case_id, run_id=run_id)

print("=== HCAT Safety Score ===")
print(f"  Fabrication rate:        {score.fabrication_rate:.1%}")
print(f"  Omission rate:           {score.omission_rate:.1%}")
print(f"  Accuracy:                {score.accuracy:.1%}")
print(f"  Verification pass rate:  {score.verification_pass_rate:.1%}")
print(f"  Evidence traceability:   {score.evidence_traceability_rate:.1%}")
print(f"  Safety score:            {score.safety_score:.1%}")
print(f"  N features:              {score.n_features}")